## Data Preprocessing for Modeling

**Purpose:** Finalize the data by transforming, encoding, and splitting it into a form suitable for machine learning models.

**Tasks:**

1. Handle missing data: Drop or impute missing values.
2. Feature scaling: Normalize or standardize features for algorithms sensitive to the scale (e.g., SVM, KNN).
3. Train-test split: Split data into training, validation, and test sets.
4. Handle categorical variables: One-hot encoding, label encoding, or other transformations for categorical features.

In [3]:
import pandas as pd
import json

data = pd.read_csv(r"D:\Doccuments\GitHub\End-to-End-MLOps-Pipeline\data\processed\CC_Data_feature_engineered.csv")

with open(r"D:\Doccuments\GitHub\End-to-End-MLOps-Pipeline\notebooks\metadata\engineered_features.json") as f:
    meta = json.load(f)

target = meta["target_col"]

In [4]:
print(data.shape)
print(data.isnull().sum().sum())
print(data[target].value_counts())

(555719, 83)
0
is_fraud
0    553574
1      2145
Name: count, dtype: int64


In [5]:
X = data.drop(columns=[target])
y = data[target]

In [6]:
split_ratio = 0.8
split_index = int(len(data) * split_ratio)

X_train = X.iloc[:split_index]
X_test  = X.iloc[split_index:]

y_train = y.iloc[:split_index]
y_test  = y.iloc[split_index:]

In [7]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns)
X_test  = pd.DataFrame(imputer.transform(X_test), columns=X.columns)

In [8]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [9]:
y_train.value_counts(normalize=True)

is_fraud
0    0.99594
1    0.00406
Name: proportion, dtype: float64

In [10]:
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
print(scale_pos_weight)

245.30193905817174


In [13]:
import os
import pickle
import json
import pandas as pd

save_path = r"D:\Doccuments\GitHub\End-to-End-MLOps-Pipeline\data\interim\fraud_data"
os.makedirs(save_path, exist_ok=True)

# Restore column names after scaling
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled  = pd.DataFrame(X_test_scaled, columns=X.columns)

# Save features
X_train_scaled.to_csv(f"{save_path}/X_train.csv", index=False)
X_test_scaled.to_csv(f"{save_path}/X_test.csv", index=False)

# Save target
y_train.to_csv(f"{save_path}/y_train.csv", index=False)
y_test.to_csv(f"{save_path}/y_test.csv", index=False)

# Save transformers
with open(f"{save_path}/imputer.pkl", "wb") as f:
    pickle.dump(imputer, f)

with open(f"{save_path}/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

feature_metadata = {
    "feature_columns": X.columns.tolist(),
    "original_num_features": X.select_dtypes(include=["int64", "float64"]).columns.tolist(),
    "num_total_features": len(X.columns),
    "target_column": "is_fraud",
    "train_shape": X_train_scaled.shape,
    "test_shape": X_test_scaled.shape,
    "target_distribution": {
        "train": y_train.value_counts().to_dict(),
        "test": y_test.value_counts().to_dict()
    }
}

with open(f"{save_path}/feature_metadata.json", "w") as f:
    json.dump(feature_metadata, f, indent=4)